# Day 060 | Mini Project: Full Geotech ML Pipeline
## 100 Days of ML for Geotechnical Engineering

**Author:** Ripon Chandra Malo  
**Phase:** 3 — Classical Machine Learning (Capstone)  
**Date:** 2026-05-16  

---

### Concept Reference Table

| Concept | Description | Geotechnical Relevance |
|---|---|---|
| Pile Bearing Capacity | Maximum load a pile can safely carry | Foundational design parameter for structures |
| Skin Friction (Shaft) | Resistance along pile shaft from soil contact | Dominant for long slender piles in clay |
| End Bearing (Tip) | Resistance at pile tip from dense soil/rock | Dominant for piles resting on hard strata |
| SPT N-Value | Standard Penetration Test blow count | Proxy for soil stiffness and strength |
| Pile Type | Driven / Bored / CFA installation method | Affects soil disturbance and capacity |
| Aspect Ratio | Length / Diameter | Governs slenderness and buckling risk |
| Feature Pipeline | End-to-end sklearn transform + model | Ensures reproducible preprocessing |
| Cross-Validation | K-fold model evaluation strategy | Prevents overfitting on limited geotech data |

---
## Part 1 | Project Problem Statement

### The Geotechnical Challenge

**Pile load tests** (static or dynamic) are the gold standard for verifying pile bearing capacity — but they cost **USD 20,000–200,000 per test** and require weeks of setup. On large infrastructure projects (bridges, wind farms, high-rises), engineers must design hundreds of piles before a single test can be run.

**The ML Opportunity:**  
We have engineering design parameters that are known *before* installation:
- Pile geometry (type, diameter, length)
- Soil conditions (type, SPT N-value, unit weight)
- Groundwater depth and layering
- Derived shaft and tip resistance estimates

**Goal:** Train a regression model that predicts pile bearing capacity (kN) from these design parameters — replacing expensive load tests with a fast, data-driven estimate.

```
Input Features         ML Model          Output
─────────────────   ─────────────   ──────────────────
pile_type           ┌─────────┐     pile_capacity_kN
pile_diameter_m  →  │ Trained │  →  (replaces load test)
pile_length_m       │  Model  │
soil_type        →  └─────────┘
N_avg, friction,
tip_resistance ...
```

**Phase 3 Capstone:** This notebook applies every skill from Days 036–059 in one complete pipeline.

---
## Part 2 | Load & Inspect the Dataset

In [ ]:
# ─────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

try:
    from xgboost import XGBRegressor
    USE_XGB = True
    print("XGBoost available — will use XGBRegressor.")
except ImportError:
    USE_XGB = False
    print("XGBoost not found — will use GradientBoostingRegressor instead.")

print("All imports successful.")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────────────────────────
df = pd.read_csv('pile_capacity_dataset.csv')

print(f"Dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]}  |  Columns: {df.shape[1]}")
print("\nColumn dtypes:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
# ─────────────────────────────────────────────────────────────────
# DESCRIPTIVE STATISTICS
# ─────────────────────────────────────────────────────────────────
print("First 5 rows:")
display(df.head())

print("\nDescriptive Statistics:")
display(df.describe())

print("\nValue counts for categorical columns:")
for col in ['pile_type', 'soil_type']:
    print(f"\n{col}:")
    print(df[col].value_counts())

---
## Part 3 | Exploratory Data Analysis (EDA)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# EDA: THREE-PANEL VISUALIZATION
# ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Pile Bearing Capacity — Exploratory Data Analysis', fontsize=14, fontweight='bold')

# --- Panel 1: Capacity Distribution ---
axes[0].hist(df['pile_capacity_kN'], bins=25, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['pile_capacity_kN'].mean(), color='red', linestyle='--', linewidth=1.5, label=f"Mean = {df['pile_capacity_kN'].mean():.0f} kN")
axes[0].axvline(df['pile_capacity_kN'].median(), color='orange', linestyle='--', linewidth=1.5, label=f"Median = {df['pile_capacity_kN'].median():.0f} kN")
axes[0].set_title('Target Distribution', fontweight='bold')
axes[0].set_xlabel('Pile Capacity (kN)')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=9)

# --- Panel 2: Capacity by Pile Type (Boxplot) ---
pile_types = df['pile_type'].unique()
data_by_type = [df[df['pile_type'] == pt]['pile_capacity_kN'].values for pt in pile_types]
bp = axes[1].boxplot(data_by_type, labels=pile_types, patch_artist=True, notch=False)
colors = ['#2196F3', '#4CAF50', '#FF9800']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Capacity by Pile Type', fontweight='bold')
axes[1].set_xlabel('Pile Type')
axes[1].set_ylabel('Pile Capacity (kN)')
axes[1].grid(axis='y', alpha=0.4)

# --- Panel 3: Capacity vs Pile Length (Scatter) ---
soil_colors = {'sand': '#FF6B35', 'clay': '#4ECDC4', 'mixed': '#95E1D3'}
for soil in df['soil_type'].unique():
    mask = df['soil_type'] == soil
    axes[2].scatter(df.loc[mask, 'pile_length_m'], df.loc[mask, 'pile_capacity_kN'],
                   c=soil_colors.get(soil, 'gray'), label=soil.title(), alpha=0.65, s=45, edgecolors='white', linewidths=0.3)
axes[2].set_title('Capacity vs Pile Length', fontweight='bold')
axes[2].set_xlabel('Pile Length (m)')
axes[2].set_ylabel('Pile Capacity (kN)')
axes[2].legend(title='Soil Type', fontsize=9)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print(f"  - Capacity ranges from {df['pile_capacity_kN'].min():.0f} to {df['pile_capacity_kN'].max():.0f} kN")
print(f"  - Driven piles show highest median capacity (more soil displacement)")
print(f"  - Clear positive correlation between pile length and capacity")
print(f"  - Sand consistently yields higher capacity than clay at same length")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CORRELATION HEATMAP (numerical features only)
# ─────────────────────────────────────────────────────────────────
num_cols = df.select_dtypes(include='number').columns.tolist()
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Pearson r')

ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels([c.replace('_', '\n') for c in num_cols], fontsize=8)
ax.set_yticklabels(num_cols, fontsize=9)

for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha='center', va='center',
                fontsize=7, color='black' if abs(corr.iloc[i, j]) < 0.7 else 'white')

ax.set_title('Correlation Matrix — Pile Capacity Dataset', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

# Top correlations with target
target_corr = corr['pile_capacity_kN'].drop('pile_capacity_kN').abs().sort_values(ascending=False)
print("\nTop feature correlations with pile_capacity_kN:")
for feat, val in target_corr.items():
    print(f"  {feat:<30} r = {val:.3f}")

---
## Part 4 | Feature Engineering

In [ ]:
# ─────────────────────────────────────────────────────────────────
# FEATURE ENGINEERING — Geotechnically Meaningful Derived Features
# ─────────────────────────────────────────────────────────────────
df_fe = df.copy()

# Geometric ratios from pile design
df_fe['aspect_ratio']   = df_fe['pile_length_m'] / df_fe['pile_diameter_m']          # slenderness
df_fe['shaft_area']     = np.pi * df_fe['pile_diameter_m'] * df_fe['pile_length_m']  # m² — shaft contact area
df_fe['tip_area']       = (np.pi / 4) * df_fe['pile_diameter_m'] ** 2               # m² — end bearing area

print("New engineered features:")
print(df_fe[['pile_diameter_m', 'pile_length_m', 'aspect_ratio', 'shaft_area', 'tip_area']].head(8).to_string(index=False))

print(f"\n  aspect_ratio  : min={df_fe['aspect_ratio'].min():.1f}  max={df_fe['aspect_ratio'].max():.1f}  mean={df_fe['aspect_ratio'].mean():.1f}")
print(f"  shaft_area    : min={df_fe['shaft_area'].min():.2f}  max={df_fe['shaft_area'].max():.2f}  mean={df_fe['shaft_area'].mean():.2f}  m²")
print(f"  tip_area      : min={df_fe['tip_area'].min():.3f}  max={df_fe['tip_area'].max():.3f}  mean={df_fe['tip_area'].mean():.3f}  m²")

# ─────────────────────────────────────────────────────────────────
# SEPARATE FEATURES AND TARGET
# ─────────────────────────────────────────────────────────────────
TARGET = 'pile_capacity_kN'
CATEGORICAL_FEATURES = ['pile_type', 'soil_type']
NUMERICAL_FEATURES = [
    'pile_diameter_m', 'pile_length_m', 'N_avg', 'shaft_friction_kPa',
    'tip_resistance_kPa', 'water_table_depth_m', 'number_of_layers',
    'avg_unit_weight_kNm3', 'aspect_ratio', 'shaft_area', 'tip_area'
]

X = df_fe[CATEGORICAL_FEATURES + NUMERICAL_FEATURES]
y = df_fe[TARGET]

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape : {y.shape}")
print(f"Total features      : {X.shape[1]}  ({len(CATEGORICAL_FEATURES)} categorical + {len(NUMERICAL_FEATURES)} numerical)")

---
## Part 5 | Train / Test Split

In [ ]:
# ─────────────────────────────────────────────────────────────────
# TRAIN / TEST SPLIT  (80 / 20, random_state=42)
# ─────────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Training set  : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set      : {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nTarget stats — Train: mean={y_train.mean():.0f} kN  std={y_train.std():.0f} kN")
print(f"Target stats — Test : mean={y_test.mean():.0f} kN  std={y_test.std():.0f} kN")
print("\nClass balance (pile_type) — Train:")
print(X_train['pile_type'].value_counts())
print("Class balance (pile_type) — Test:")
print(X_test['pile_type'].value_counts())

---
## Part 6 | sklearn Pipeline with ColumnTransformer

In [ ]:
# ─────────────────────────────────────────────────────────────────
# PREPROCESSING PIPELINE
# Numerical  → StandardScaler  (zero mean, unit variance)
# Categorical → OrdinalEncoder  (label-encodes pile_type, soil_type)
# ─────────────────────────────────────────────────────────────────
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('encoder', OrdinalEncoder(
        categories=[
            ['driven', 'bored', 'CFA'],
            ['sand', 'mixed', 'clay']
        ],
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, CATEGORICAL_FEATURES),
        ('num', numerical_transformer,  NUMERICAL_FEATURES),
    ]
)

print("Preprocessing pipeline structure:")
print("  ColumnTransformer")
print("    ├── categorical → OrdinalEncoder   [pile_type, soil_type]")
print("    └── numerical   → StandardScaler   [11 numerical features]")
print(f"\n  Output dimensionality: {len(CATEGORICAL_FEATURES) + len(NUMERICAL_FEATURES)} features total")

---
## Part 7 | Fit & Compare Four Models (5-Fold Cross-Validation)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# DEFINE FOUR MODEL PIPELINES
# ─────────────────────────────────────────────────────────────────
boosted_model = XGBRegressor(n_estimators=100, random_state=42, verbosity=0) if USE_XGB else \
                GradientBoostingRegressor(n_estimators=100, random_state=42)
boosted_name = 'XGBRegressor' if USE_XGB else 'GradientBoostingRegressor'

models = {
    'LinearRegression'     : Pipeline([('pre', preprocessor), ('mdl', LinearRegression())]),
    'RandomForestRegressor': Pipeline([('pre', preprocessor), ('mdl', RandomForestRegressor(n_estimators=100, random_state=42))]),
    boosted_name           : Pipeline([('pre', preprocessor), ('mdl', boosted_model)]),
    'SVR'                  : Pipeline([('pre', preprocessor), ('mdl', SVR(kernel='rbf', C=100, epsilon=50))]),
}

print("Running 5-fold Cross-Validation on training set...")
print("-" * 58)
print(f"  {'Model':<32} {'CV R² (mean)':>12} {'CV R² (std)':>12}")
print("-" * 58)

cv_results = {}
for name, pipe in models.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)
    cv_results[name] = scores
    print(f"  {name:<32} {scores.mean():>12.4f} {scores.std():>12.4f}")

print("-" * 58)
best_model_name = max(cv_results, key=lambda k: cv_results[k].mean())
print(f"\n  Best model by CV R²: {best_model_name}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# VISUALISE CV COMPARISON
# ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

names  = list(cv_results.keys())
means  = [cv_results[n].mean() for n in names]
stds   = [cv_results[n].std()  for n in names]
colors_bar = ['#4472C4', '#ED7D31', '#A9D18E', '#FF6B6B']

bars = ax.barh(names, means, xerr=stds, color=colors_bar, alpha=0.82,
               edgecolor='white', height=0.55, capsize=5)

for bar, mean in zip(bars, means):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"R² = {mean:.4f}", va='center', fontsize=10, fontweight='bold')

ax.set_xlim(0, 1.12)
ax.set_xlabel('5-Fold Cross-Validation R²', fontsize=11)
ax.set_title('Model Comparison — Pile Capacity Prediction', fontsize=13, fontweight='bold')
ax.axvline(0.8, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='R² = 0.80 threshold')
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 8 | Hyperparameter Tuning (RandomizedSearchCV)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# HYPERPARAMETER TUNING — RandomForest (best tree model for tuning)
# We always tune RandomForest as it is the most interpretable tree
# ensemble, regardless of whether XGB scores marginally higher.
# ─────────────────────────────────────────────────────────────────
param_dist = {
    'mdl__n_estimators'      : [50, 100, 150, 200, 300],
    'mdl__max_depth'         : [None, 5, 8, 12, 16],
    'mdl__min_samples_split' : [2, 4, 6, 8],
    'mdl__min_samples_leaf'  : [1, 2, 3, 4],
    'mdl__max_features'      : ['sqrt', 'log2', 0.6, 0.8],
    'mdl__bootstrap'         : [True, False],
}

rf_pipe = Pipeline([
    ('pre', preprocessor),
    ('mdl', RandomForestRegressor(random_state=42))
])

search = RandomizedSearchCV(
    rf_pipe,
    param_distributions=param_dist,
    n_iter=40,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)

print(f"\nBest CV R² : {search.best_score_:.4f}")
print("Best hyperparameters:")
for k, v in search.best_params_.items():
    print(f"  {k:<35} = {v}")

best_pipeline = search.best_estimator_

---
## Part 9 | Final Evaluation on Test Set

In [ ]:
# ─────────────────────────────────────────────────────────────────
# EVALUATE BEST MODEL ON HELD-OUT TEST SET
# ─────────────────────────────────────────────────────────────────
y_pred = best_pipeline.predict(X_test)

r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)

print("=" * 48)
print("  FINAL TEST SET PERFORMANCE")
print("  Best model: Tuned RandomForestRegressor")
print("=" * 48)
print(f"  R²   = {r2:.4f}     (explains {r2*100:.1f}% of capacity variance)")
print(f"  RMSE = {rmse:.1f} kN  (typical error in absolute terms)")
print(f"  MAE  = {mae:.1f} kN  (average absolute error)")
print("=" * 48)

rel_error_pct = mae / y_test.mean() * 100
print(f"\n  Mean pile capacity (test) : {y_test.mean():.0f} kN")
print(f"  MAE as % of mean capacity : {rel_error_pct:.1f}%")
print(f"  (A static load test costs ~USD 50,000+ — this model provides")
print(f"   a {rel_error_pct:.0f}% error estimate at near-zero marginal cost)")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# DIAGNOSTIC PLOTS: Actual vs Predicted + Residuals
# ─────────────────────────────────────────────────────────────────
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Tuned RandomForestRegressor — Test Set Diagnostics', fontsize=13, fontweight='bold')

# --- Panel 1: Actual vs Predicted ---
vmin = min(y_test.min(), y_pred.min())
vmax = max(y_test.max(), y_pred.max())
axes[0].scatter(y_test, y_pred, alpha=0.65, c='steelblue', edgecolors='white',
                linewidths=0.3, s=50, label='Test samples')
axes[0].plot([vmin, vmax], [vmin, vmax], 'r--', linewidth=2, label='1:1 line (perfect fit)')
axes[0].fill_between([vmin, vmax],
                     [vmin - 1.96*rmse, vmax - 1.96*rmse],
                     [vmin + 1.96*rmse, vmax + 1.96*rmse],
                     alpha=0.1, color='red', label='±1.96 × RMSE band')
axes[0].set_xlabel('Actual Pile Capacity (kN)', fontsize=11)
axes[0].set_ylabel('Predicted Pile Capacity (kN)', fontsize=11)
axes[0].set_title(f'Actual vs Predicted  (R² = {r2:.4f})', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)
axes[0].text(0.05, 0.93, f'RMSE = {rmse:.0f} kN\nMAE  = {mae:.0f} kN',
             transform=axes[0].transAxes, fontsize=9,
             bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.8))

# --- Panel 2: Residual Plot ---
axes[1].scatter(y_pred, residuals, alpha=0.65, c='#FF6B6B', edgecolors='white',
                linewidths=0.3, s=50)
axes[1].axhline(0, color='black', linewidth=1.5, linestyle='-')
axes[1].axhline( 2*rmse, color='gray', linewidth=1, linestyle='--', alpha=0.6, label=f'+2σ = {2*rmse:.0f} kN')
axes[1].axhline(-2*rmse, color='gray', linewidth=1, linestyle='--', alpha=0.6, label=f'-2σ = {-2*rmse:.0f} kN')
axes[1].set_xlabel('Predicted Pile Capacity (kN)', fontsize=11)
axes[1].set_ylabel('Residual (Actual − Predicted) [kN]', fontsize=11)
axes[1].set_title('Residual Plot', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 10 | Feature Importance

In [ ]:
# ─────────────────────────────────────────────────────────────────
# FEATURE IMPORTANCE — from Tuned Random Forest
# ─────────────────────────────────────────────────────────────────
rf_model   = best_pipeline.named_steps['mdl']
feat_names = CATEGORICAL_FEATURES + NUMERICAL_FEATURES
importances = rf_model.feature_importances_

feat_imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances})
feat_imp_df = feat_imp_df.sort_values('importance', ascending=False)

print("Feature Importances (Mean Decrease in Impurity):")
print("-" * 48)
for _, row in feat_imp_df.iterrows():
    bar = '█' * int(row['importance'] * 300)
    print(f"  {row['feature']:<28} {row['importance']:.4f}  {bar}")

# ─── Visualise ───
fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = ['#2196F3' if i < 3 else '#90CAF9' for i in range(len(feat_imp_df))]
ax.barh(feat_imp_df['feature'][::-1], feat_imp_df['importance'][::-1],
        color=bar_colors[::-1], alpha=0.85, edgecolor='white')
ax.set_xlabel('Feature Importance (MDI)', fontsize=11)
ax.set_title('Random Forest Feature Importances\nPile Capacity Prediction', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

top3 = feat_imp_df.head(3)['feature'].tolist()
ax.text(0.98, 0.02, f'Top 3: {top3}', transform=ax.transAxes,
        ha='right', va='bottom', fontsize=8,
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print(f"\nGeotechnical insight:")
print(f"  Top features: {top3}")
print(f"  This aligns with pile design theory — shaft area and length")
print(f"  are primary drivers; soil strength (N_avg) is secondary.")

---
## Part 11 | Predict Capacity for New Pile Designs

In [ ]:
# ─────────────────────────────────────────────────────────────────
# PREDICT CAPACITY FOR 3 NEW PILE DESIGNS
# These represent real design scenarios an engineer might evaluate.
# ─────────────────────────────────────────────────────────────────
new_designs_raw = [
    # Scenario 1: Large driven pile in dense sand (bridge foundation)
    {'pile_type': 'driven', 'pile_diameter_m': 0.90, 'pile_length_m': 30.0,
     'soil_type': 'sand', 'N_avg': 48, 'shaft_friction_kPa': 112.0,
     'tip_resistance_kPa': 4700.0, 'water_table_depth_m': 2.0,
     'number_of_layers': 4, 'avg_unit_weight_kNm3': 20.2},

    # Scenario 2: Medium bored pile in clay (building foundation)
    {'pile_type': 'bored',  'pile_diameter_m': 0.80, 'pile_length_m': 22.0,
     'soil_type': 'clay', 'N_avg': 25, 'shaft_friction_kPa': 58.0,
     'tip_resistance_kPa': 1450.0, 'water_table_depth_m': 3.5,
     'number_of_layers': 3, 'avg_unit_weight_kNm3': 18.1},

    # Scenario 3: Small CFA pile in mixed soil (wind turbine)
    {'pile_type': 'CFA',    'pile_diameter_m': 0.50, 'pile_length_m': 18.0,
     'soil_type': 'mixed', 'N_avg': 33, 'shaft_friction_kPa': 67.0,
     'tip_resistance_kPa': 2300.0, 'water_table_depth_m': 3.0,
     'number_of_layers': 3, 'avg_unit_weight_kNm3': 19.2},
]

# ─────────────────────────────────────────────────────────────────
# Add derived features before prediction
# ─────────────────────────────────────────────────────────────────
new_designs_df = pd.DataFrame(new_designs_raw)
new_designs_df['aspect_ratio'] = new_designs_df['pile_length_m'] / new_designs_df['pile_diameter_m']
new_designs_df['shaft_area']   = np.pi * new_designs_df['pile_diameter_m'] * new_designs_df['pile_length_m']
new_designs_df['tip_area']     = (np.pi / 4) * new_designs_df['pile_diameter_m'] ** 2

X_new = new_designs_df[CATEGORICAL_FEATURES + NUMERICAL_FEATURES]
y_new_pred = best_pipeline.predict(X_new)

scenarios = ['Bridge Foundation (Driven/Sand)', 'Building Foundation (Bored/Clay)', 'Wind Turbine (CFA/Mixed)']

print("=" * 65)
print("  PILE CAPACITY PREDICTIONS — NEW DESIGN SCENARIOS")
print("=" * 65)
for i, (scenario, pred) in enumerate(zip(scenarios, y_new_pred)):
    d = new_designs_raw[i]
    print(f"\n  Scenario {i+1}: {scenario}")
    print(f"    Pile    : {d['pile_type'].upper()} | ⌀{d['pile_diameter_m']:.2f} m | L={d['pile_length_m']:.1f} m")
    print(f"    Soil    : {d['soil_type'].title()} | N_avg={d['N_avg']} | WT depth={d['water_table_depth_m']:.1f} m")
    print(f"    PREDICTED CAPACITY : {pred:,.0f} kN  ({pred/1000:.2f} MN)")
    safe_load = pred / 2.5
    print(f"    Safe working load  : {safe_load:,.0f} kN  (FoS = 2.5)")
print("\n" + "=" * 65)

---
## Part 12 | Phase 3 Summary — All 25 Topics

In [ ]:
# ─────────────────────────────────────────────────────────────────
# PHASE 3 SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────
phase3_topics = [
    ('036', 'Intro to Classical ML',           'sklearn Estimator API'),
    ('037', 'Linear Regression',               'OLS / Gradient Descent'),
    ('038', 'Polynomial Regression',           'PolynomialFeatures + Pipeline'),
    ('039', 'Ridge & Lasso Regression',        'L2 / L1 Regularization'),
    ('040', 'Logistic Regression',             'Sigmoid, Log-loss'),
    ('041', 'K-Nearest Neighbours',            'Distance-weighted voting'),
    ('042', 'Naive Bayes',                     'Bayes theorem + class priors'),
    ('043', 'Decision Trees',                  'Gini / Entropy splitting'),
    ('044', 'Bagging & Random Forest',         'Bootstrap aggregation'),
    ('045', 'Gradient Boosting',               'Additive residual fitting'),
    ('046', 'XGBoost',                         'Regularised boosting + tree pruning'),
    ('047', 'LightGBM & CatBoost',             'Leaf-wise growth, native categoricals'),
    ('048', 'Support Vector Machines',         'Margin maximisation, kernel trick'),
    ('049', 'Feature Selection',               'Filter / Wrapper / Embedded'),
    ('050', 'Feature Engineering',             'Domain-driven transforms'),
    ('051', 'Cross-Validation',                'K-Fold, Stratified, LOOCV'),
    ('052', 'Bias-Variance Trade-off',         'Overfitting vs underfitting'),
    ('053', 'Hyperparameter Tuning',           'Grid / Random / Bayesian Search'),
    ('054', 'Model Evaluation Metrics',        'R², RMSE, MAE, F1, AUC'),
    ('055', 'Imbalanced Datasets',             'SMOTE, class_weight, thresholds'),
    ('056', 'Dimensionality Reduction (PCA)',  'Eigen decomposition, explained variance'),
    ('057', 'Clustering (K-Means)',            'Elbow method, Silhouette score'),
    ('058', 'Ensemble Methods',               'Voting, Stacking, Blending'),
    ('059', 'Pipeline & ColumnTransformer',    'Reproducible preprocessing chains'),
    ('060', 'Mini Project — Full Pipeline',    'End-to-end pile capacity prediction'),
]

print(f"  {'Day':<6} {'Topic':<40} {'Key Technique':<35}")
print("  " + "-" * 81)
for day, topic, technique in phase3_topics:
    marker = '  <<< TODAY' if day == '060' else ''
    print(f"  {day:<6} {topic:<40} {technique:<35}{marker}")

print("\n  Phase 3 COMPLETE — 25 classical ML topics mastered.")

---
### Phase 3 Quick-Reference Table

| Day | Topic | Key Technique |
|:---:|---|---|
| 036 | Intro to Classical ML | sklearn Estimator API |
| 037 | Linear Regression | OLS / Gradient Descent |
| 038 | Polynomial Regression | PolynomialFeatures + Pipeline |
| 039 | Ridge & Lasso | L2 / L1 Regularization |
| 040 | Logistic Regression | Sigmoid, Log-loss |
| 041 | KNN | Distance-weighted voting |
| 042 | Naive Bayes | Bayes theorem + class priors |
| 043 | Decision Trees | Gini / Entropy splitting |
| 044 | Bagging & Random Forest | Bootstrap aggregation |
| 045 | Gradient Boosting | Additive residual fitting |
| 046 | XGBoost | Regularised boosting |
| 047 | LightGBM & CatBoost | Leaf-wise growth |
| 048 | SVM | Margin maximisation, kernel trick |
| 049 | Feature Selection | Filter / Wrapper / Embedded |
| 050 | Feature Engineering | Domain-driven transforms |
| 051 | Cross-Validation | K-Fold, Stratified, LOOCV |
| 052 | Bias-Variance Trade-off | Overfitting vs underfitting |
| 053 | Hyperparameter Tuning | Grid / Random / Bayesian Search |
| 054 | Model Evaluation Metrics | R², RMSE, MAE, F1, AUC |
| 055 | Imbalanced Datasets | SMOTE, class_weight |
| 056 | PCA | Eigen decomposition |
| 057 | Clustering | K-Means, Silhouette score |
| 058 | Ensemble Methods | Voting, Stacking, Blending |
| 059 | Pipeline & ColumnTransformer | Reproducible preprocessing |
| **060** | **Mini Project — Full Pipeline** | **End-to-end geotech ML** |

---
## Part 13 | Phase 4 Preview — Deep Learning for Geotechnics

| Day | Topic | Geotechnical Application |
|:---:|---|---|
| 061 | Intro to Neural Networks | Universal approximators for non-linear soil behaviour |
| 062 | Perceptron & Backpropagation | Learning soil classification boundaries |
| 063 | Activation Functions | Capturing non-linearity in stress-strain relationships |
| 064 | Keras / PyTorch Basics | Building geo-ML models with deep learning frameworks |
| 065 | Regularisation in DL | Dropout / BatchNorm for small geotech datasets |
| 066 | CNN for Geotechnical Images | Soil particle size analysis from micro-CT scans |
| 067 | RNN / LSTM | Time-series piezometer and settlement monitoring |
| 068 | Autoencoders | Anomaly detection in sensor data |
| 069 | Transfer Learning | Reusing pre-trained models for geotech image tasks |
| 070 | DL Mini Project | Settlement prediction from CPT time series |

> **Coming Up:** Phase 4 introduces deep learning — moving beyond classical ML into neural networks that can model highly non-linear geotechnical behaviour, learn from images, and handle sequential monitoring data.

---
## Practice Exercises

Try these to deepen your understanding:

In [ ]:
# ─────────────────────────────────────────────────────────────────
# PRACTICE EXERCISES (uncomment and complete each)
# ─────────────────────────────────────────────────────────────────

# Exercise 1:
# Add a new derived feature: 'capacity_per_meter' (rough estimate)
# = shaft_friction_kPa * shaft_area / pile_length_m
# Retrain and check if R² improves.
# df_fe['capacity_per_meter'] = ...

# Exercise 2:
# Compare RandomForestRegressor vs GradientBoostingRegressor
# by plotting their learning curves (train vs CV R² as n_estimators increases).
# from sklearn.model_selection import learning_curve
# ...

# Exercise 3:
# Instead of OrdinalEncoder, try OneHotEncoding the categorical features.
# from sklearn.preprocessing import OneHotEncoder
# Does linear regression improve significantly with OHE?
# ...

# Exercise 4:
# Build a simple stacking ensemble:
# Level 0: RF + GBM, Level 1: LinearRegression
# from sklearn.ensemble import StackingRegressor
# Compare stacked R² against tuned RF alone.
# ...

# Exercise 5:
# Use SHAP to explain individual pile capacity predictions.
# import shap
# explainer = shap.TreeExplainer(rf_model)
# shap_values = explainer.shap_values(X_test_transformed)
# shap.summary_plot(shap_values, X_test, feature_names=feat_names)

print("Practice exercises loaded — uncomment and run each block to explore.")

---
## ML Connection — Techniques Used in This Project

| ML Technique | Where Applied | Why It Matters |
|---|---|---|
| OrdinalEncoder | pile_type, soil_type → numeric | Tree models don't need one-hot; ordinal is faster |
| StandardScaler | All 11 numerical features | Required for SVR, helps LR convergence |
| ColumnTransformer | Apply different transforms per column type | Clean, leak-proof preprocessing |
| sklearn Pipeline | Chain preprocessing + model | Guarantees test data processed identically to train |
| Cross-Validation (5-fold) | Model selection stage | Stable R² estimate with 180 samples |
| RandomizedSearchCV | RandomForest tuning | 40 iterations explores hyperparameters efficiently |
| Feature Engineering | aspect_ratio, shaft_area, tip_area | Physics-based features improve tree splitting |
| R² / RMSE / MAE | Final evaluation | Multi-metric assessment avoids misleading conclusions |
| Feature Importance (MDI) | Model interpretation | Identifies which design parameters matter most |
| Residual Analysis | Diagnostic plotting | Checks for systematic bias across capacity range |

---
## Key Takeaways

1. **A complete ML pipeline includes six stages:** data loading, EDA, feature engineering, preprocessing, model selection (with CV), and final evaluation — skipping any stage risks unreliable models in geotechnical practice.

2. **sklearn Pipelines are non-negotiable for geotech ML:** they guarantee that StandardScaler is fit on training data only and applied identically to test/deployment data — preventing data leakage that would inflate R².

3. **Domain knowledge drives better features:** `shaft_area`, `tip_area`, and `aspect_ratio` are rooted in pile design theory (Meyerhof, API) — these engineered features typically outperform raw geometry in tree-based models.

4. **RandomForest is robust for small geotech datasets:** with only 180 samples it achieves strong R² through bootstrap aggregation and is naturally resistant to outliers from site variability.

5. **Model uncertainty matters as much as accuracy:** in pile design, an RMSE of 150 kN on a 1,500 kN pile (10%) directly affects the Factor of Safety — always report prediction intervals alongside point estimates.

---

*Day 060 of 100 — Phase 3 Complete!*

*Classical ML fully mastered — the foundation is solid. Phase 4 awaits: deep learning, neural networks, and the next frontier of geotechnical AI.*